<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">Lab 03: トラベルエージェントにFunction Toolsを追加する</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Microsoft Foundry Agent Observability
  </p>
</div>

## 学習内容

このラボでは、Contoso Travel エージェントに **Function Tools** を使って**実データを検索する**能力を与えます。フライト・ホテル・レンタカーのCSVデータを検索するPython関数を定義し、エージェントが呼び出せるツールとして登録します。これにより、コンシェルジュは汎用チャットボットからデータ駆動型の旅行アシスタントへと生まれ変わります。

ツール呼び出しのフローは次のようになります:

```
User Query → Agent (LLM decides to call a tool)
                → function_call request
                    → Python function executes
                        → results returned to Agent
                            → Agent synthesizes final answer
```

---

## 2. セットアップ

---


In [ ]:
# セットアップ: FunctionTool（ツールの登録用）と
# Tool（基底型のリストヒント）のインポートを含む
import os
import json
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool

# .env は notebooks ディレクトリの一つ上の階層にある
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

tenant_id = os.getenv("TENANT_ID")
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = AzureCliCredential(tenant_id=tenant_id)
#credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Microsoft Foundry に接続しました")

## 3. 旅行データの読み込み

旅行 CSV ファイルを pandas DataFrame に読み込みます。ツール関数はこれらの DataFrame をクエリします。

---


In [ ]:
# パスはノートブックの場所からの相対パス:
# notebooks/1-prompt-agents/ -> ../../data/
data_path = "../../data/contoso-travel"

flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"✈️  {len(flights_df)} 件のフライトを読み込みました")
print(f"🏨 {len(hotels_df)} 件のホテルを読み込みました")
print(f"🚗 {len(cars_df)} 件のレンタカーを読み込みました")

## 4. ツール関数の定義

各関数はクエリパラメーターに基づいて旅行データを検索し、マッチした結果を JSON として返します。

---


In [ ]:
# ツール関数は JSON 文字列（辞書ではなく）を返す必要がある。
# エージェントはツール出力としてその生の文字列を受け取る。

def search_flights(origin: str = None, destination: str = None, cabin_class: str = None, max_price: float = None) -> str:
    """Search for available flights based on criteria."""
    results = flights_df.copy()  # ソースの変更を防ぐ
    if origin:
        # ユーザーの柔軟な入力のために大文字小文字を区別しないマッチング
        results = results[results["origin"].str.lower() == origin.lower()]
    if destination:
        results = results[results["destination"].str.lower() == destination.lower()]
    if cabin_class:
        results = results[results["cabin_class"].str.lower() == cabin_class.lower()]
    if max_price:
        results = results[results["price_usd"] <= max_price]
    
    if results.empty:
        return json.dumps({"message": "No flights found matching your criteria.", "results": []})
    
    # orient="records" -> 辞書のリスト形式、LLM にとって最も扱いやすい
    return results.to_json(orient="records")


def search_hotels(city: str = None, min_stars: int = None, max_price: float = None, amenities: str = None) -> str:
    """Search for available hotels based on criteria."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if min_stars:
        results = results[results["star_rating"] >= min_stars]
    if max_price:
        results = results[results["price_per_night_usd"] <= max_price]
    if amenities:
        # 部分一致: "Pool" は "Pool, Spa, WiFi" にマッチする
        results = results[results["amenities"].str.lower().str.contains(amenities.lower())]
    
    if results.empty:
        return json.dumps({"message": "No hotels found matching your criteria.", "results": []})
    
    return results.to_json(orient="records")


def search_car_rentals(city: str = None, car_type: str = None, max_price: float = None) -> str:
    """Search for available car rentals based on criteria."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    if max_price:
        results = results[results["price_per_day_usd"] <= max_price]
    # 常に利用可能なもののみフィルタリング（フライト・ホテルとは異なる）
    results = results[results["available"] == True]
    
    if results.empty:
        return json.dumps({"message": "No car rentals found matching your criteria.", "results": []})
    
    return results.to_json(orient="records")


# 動作確認テスト
print("🧪 search_flights('Seattle', 'Paris') のテスト:")
print(search_flights(origin="Seattle", destination="Paris"))

## 5. Function Toolsの登録

各関数が受け付けるパラメーターをエージェントに伝える JSON スキーマを持つ `FunctionTool` 定義を作成します。

---


In [ ]:
# FunctionTool: JSON スキーマにより、エージェントが各関数に
# 渡すパラメーターを把握する。エージェントはツール呼び出しを
# 決定した際に、このスキーマに準拠した引数を生成する。

flight_tool = FunctionTool(
    name="search_flights",
    description="Search for available flights. Use this when the customer asks about flights, airfare, or flying to a destination.",
    parameters={
        "type": "object",
        "properties": {
            "origin": {"type": "string", "description": "Departure city name"},
            "destination": {"type": "string", "description": "Arrival city name"},
            # enum でモデルが生成できる値を制約する
            "cabin_class": {"type": "string", "description": "Cabin class: Economy, Business, or First", "enum": ["Economy", "Business", "First"]},
            "max_price": {"type": "number", "description": "Maximum ticket price in USD"},
        },
        "required": [],  # 柔軟なクエリのため全パラメーターをオプションに
        "additionalProperties": False,
    },
    strict=False,  # オプションパラメーターを許可（strict は全必須）
)

hotel_tool = FunctionTool(
    name="search_hotels",
    description="Search for available hotels. Use this when the customer asks about hotels, accommodation, or places to stay.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name to search hotels in"},
            "min_stars": {"type": "integer", "description": "Minimum star rating (1-5)"},
            "max_price": {"type": "number", "description": "Maximum price per night in USD"},
            "amenities": {"type": "string", "description": "Desired amenity to filter by (e.g., Pool, Spa, WiFi)"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

car_rental_tool = FunctionTool(
    name="search_car_rentals",
    description="Search for available car rentals. Use this when the customer asks about rental cars or ground transportation.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name for car pickup"},
            "car_type": {"type": "string", "description": "Type of car: Economy, SUV, Luxury, or Minivan", "enum": ["Economy", "SUV", "Luxury", "Minivan"]},
            "max_price": {"type": "number", "description": "Maximum price per day in USD"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

tools: list[Tool] = [flight_tool, hotel_tool, car_rental_tool]
print(f"✅ {len(tools)} 個の Function Tool を定義しました: {[t.name for t in tools]}")

## 6. トラベルエージェントの作成

Function Tools を付加した状態でエージェントを作成します。

---


In [ ]:
# トラベルエージェントのシステム指示を定義する
TRAVEL_AGENT_INSTRUCTIONS = """あなたは Contoso Travel エージェントです。リアルタイムの旅行データにアクセスできる、旅行の専門アシスタントです。

以下のツールを利用できます：
- search_flights: 都市間のフライトを検索する
- search_hotels: 目的地の都市のホテルを検索する
- search_car_rentals: 都市のレンタカーオプションを検索する

お客様が旅行オプションについて尋ねたとき：
1. 適切なツールを使用して在庫を検索してください
2. 結果を明確で整理された形式で提示してください
3. 価格、評価、空き状況などの関連情報を含めてください
4. 結果に基づいて役立つ推薦を提供してください

常に丁寧で正確、プロフェッショナルな対応を心がけてください。条件に合う結果がない場合は、別の検索条件を提案してください。"""

# tools= で FunctionTool 定義を付加し、エージェントが
# レスポンス中にどの関数を呼び出せるかを知らせる
agent = project_client.agents.create_version(
    agent_name="contoso-travel-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=TRAVEL_AGENT_INSTRUCTIONS,
        tools=tools,
    ),
)

print(f"✅ {len(tools)} 個のツールを持つトラベルエージェントを作成しました!")
print(f"   名前: {agent.name}")
print(f"   バージョン: {agent.version}")

## 7. テスト: フライト検索

フライト検索クエリでエージェントをテストします。エージェントは `search_flights` Function Tool を呼び出します。

---


In [ ]:
# FunctionCallOutput: ツールの結果をモデルに返すために
# 使用される型（モデルが最終的な回答を合成するため）
from openai.types.responses.response_input_param import FunctionCallOutput

def handle_tool_calls(response):
    """Process any function_call output items and return results."""
    # ツール名をローカルの Python 関数にマッピング
    tool_map = {
        "search_flights": search_flights,
        "search_hotels": search_hotels,
        "search_car_rentals": search_car_rentals,
    }
    
    function_results = []
    for item in response.output:
        # "function_call" アイテム = エージェントがツールを要求している
        if item.type == "function_call":
            func = tool_map.get(item.name)
            if func:
                # arguments は辞書ではなく JSON 文字列
                args = json.loads(item.arguments)
                print(f"   🔧 {item.name}({args}) を呼び出し中")
                result = func(**args)
                # call_id でこの結果をリクエストに紐付ける
                function_results.append(
                    FunctionCallOutput(
                        type="function_call_output",
                        call_id=item.call_id,
                        output=result,
                    )
                )
    return function_results

# 新しい会話を開始してツール呼び出しをトリガーする
conversation = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="シアトルからパリへの800ドル以下のフライトを探してください。",
)

# レスポンスにはテキストではなく function_call アイテムが含まれている
print("📨 エージェントがツール呼び出しを要求しました:")
function_results = handle_tool_calls(response)

## 8. 関数呼び出しレスポンスの処理

エージェントが関数呼び出しを発行しました。関数を実行して結果を返し、エージェントが最終的な回答を生成できるようにします。

---


In [ ]:
# ツールの結果を返してエージェントが生のデータから
# 自然言語の回答を生成できるようにする。
# previous_response_id で前のターンに連鎖させる。
if function_results:
    response = openai_client.responses.create(
        input=function_results,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

print(f"\n🤖 トラベルエージェントの回答:\n")
print(response.output_text)

## 9. テスト: ホテル＋レンタカー

複数のツール呼び出しをトリガーする複雑なクエリをテストします。

---


In [ ]:
# マルチツール: エージェントは一度にすべてではなく、
# ラウンドをまたいでツールを順次呼び出す場合がある
# （例: まずホテルを検索し、次にレンタカーを検索）
conversation2 = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation2.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="東京で4つ星ホテルとレンタカーが必要です。ホテルの予算は1泊200ドルです。",
)

print("📨 エージェントがツール呼び出しを要求しました:")
function_results = handle_tool_calls(response)

if function_results:
    # 最初のバッチのツール結果を返す
    response = openai_client.responses.create(
        input=function_results,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

    # エージェントが第2ラウンドのツール呼び出しを発行する場合がある
    # （例: まずホテルを検索し、次にレンタカーが必要になる）
    more_results = handle_tool_calls(response)
    if more_results:
        response = openai_client.responses.create(
            input=more_results,
            previous_response_id=response.id,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )

print(f"\n🤖 トラベルエージェントの回答:\n")
print(response.output_text)

## 10. クリーンアップ

---


In [ ]:
# クリーンアップ: 両方の会話とエージェントバージョンを削除する
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ 会話を削除しました")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ エージェントバージョンを削除しました")

openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントを閉じました")